<a href="https://colab.research.google.com/github/whtan88/RandomDataAnalyst_DataAnalysis/blob/main/I_Let_AI_find_the_Best_Model_for_Stock_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install uv
!uv pip install --system autogluon --no-cache

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 26.7 MB/s  0:00:00
Using Python 3.13.18 environment at: /usr
Resolved 258 packages in 15.34s
Prepared 101 packages in 1m 18s
Uninstalled 19 packages in 2.24s
Installed 101 packages in 1.52s
 + adagio==0.2.6
 + aiohttp-cors==0.8.1
 + alembic==1.20.1
 + antlr4-python3-runtime==4.9.3
 + apswutils==0.1.2
 + autogluon==1.6.3
 + autogluon-common==1.6.3
 + autogluon-core==1.6.3
 + autogluon-features==1.6.3
 + autogluon-multimodal==1.6.3
 + autogluon-tabular==1.6.3
 + autogluon-timeseries==1.6.3
 + boto3==1.44.5
 + botocore==1.44.5
 + catboost==1.2.10
 + chronos-forecasting==2.3.2
 + colorama==0.4.6
 + colorful==0.5.8
 + colorlog==6.12.0
 + coreforecast==0.0.16
 + cuda-bindings==13.5.0
 + cuda-pathfinder==1.9.0
 + cuda-toolkit==13.0.3.0
 + distlib==0.4.4
 + einx==0.4.3
 + evaluate==0.4.6
 - fastai==2.8.12
 + fastai==2.8.7
 - fastcore==2.2.37
 + fastcore==1.14.5
 + fastlite==0.2.4
 - fasttransform==0.0.5
 + fasttransform==0.0.2
 + fugue==

In [ ]:
!pip install pandas-ta-classic
!pip install catboost

# Import libraries
import pandas_ta_classic as ta

from autogluon.tabular import TabularDataset, TabularPredictor

import re
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
from datetime import datetime,timedelta
from sklearn.preprocessing import StandardScaler # This should now work
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import math
from collections import Counter

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.calibration import CalibratedClassifierCV, calibration_curve, CalibrationDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report, roc_curve, roc_auc_score

In [ ]:
# Function
def feature_engineering_1(df,ticker):
  # Insert Feature engineering here
  return df

def feature_engineering_2(df, ticker):
  # Insert Feature engineering here
  return df

def prep_data(ticker, start_date, test_start_date, current_date, label='binary'):

  def flatten_multiindex_columns(df_to_flatten):
      new_columns = []
      for col in df_to_flatten.columns:
          if isinstance(col, tuple):
              clean_parts = [str(part) for part in col if str(part).strip()]
              if clean_parts:
                  new_columns.append('_'.join(clean_parts))
              else:
                  if len(col) > 0 and str(col[0]).strip():
                      new_columns.append(str(col[0]))
                  else:
                      new_columns.append('_'.join(map(str, col)))
          else:
              new_columns.append(str(col))
      df_to_flatten.columns = new_columns
      return df_to_flatten

  df = yf.download(ticker, start=start_date, end=current_date, interval='1d', auto_adjust=True).reset_index()
  df = df.sort_index()
  # Flatten columns
  df = flatten_multiindex_columns(df)
  # Rename columns to remove ticker for easier access with pandas_ta functions
  df.columns = [col.replace(f'_{ticker}', '') for col in df.columns]

  df = feature_engineering_1(df,ticker)
  df = feature_engineering_2(df,ticker)

  # Drop rows with NaN values after feature engineering but before splitting
  df.dropna(inplace=True)


  # Corrected list_drop to match the actual DataFrame column names
  list_drop = ['target_close', 'Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'cat_close', 'SMA_5', 'SMA_10', 'SMA_20', 'EMA_5', 'EMA_10', 'EMA_20']

  # Reconstruct list_drop with flattened names
  list_drop_flattened = []
  for item in list_drop:
      if isinstance(item, tuple):
          clean_parts = [str(part) for part in item if str(part).strip()]
          if clean_parts:
              list_drop_flattened.append('_'.join(clean_parts))
          else:
              if len(item) > 0 and str(item[0]).strip():
                  list_drop_flattened.append(str(item[0]))
              else:
                  list_drop_flattened.append('_'.join(map(str, item)))
      else:
          list_drop_flattened.append(str(item))

  # Split dataframes based on the cleaned df
  # Use flattened column names
  df_train_raw = df[df['Date'] < test_start_date].copy()
  df_test_raw = df[df['Date'] >= test_start_date][:-1].copy() # Exclude the very last row for prediction
  df_predict_raw = df.iloc[-1:].copy()

  #print(df_test_raw['Date']) # Use flattened column name

  # Prepare target labels y_train and y_test from 'cat_close' before it's dropped from features
  y_train = df_train_raw['cat_close']
  y_test = df_test_raw['cat_close']

  # Extract the list of dates
  test_date = df_test_raw['Date']
  list_date = df_predict_raw['Date']

  # Drop non-feature columns from the dataframes BEFORE scaling
  # Use the flattened list_drop
  df_train = df_train_raw.drop(columns=list_drop_flattened)
  df_test = df_test_raw.drop(columns=list_drop_flattened)
  df_predict = df_predict_raw.drop(columns=list_drop_flattened)

  scaler = StandardScaler()

  # Apply scaler to the processed dataframes
  df_train_scaled = scaler.fit_transform(df_train)
  df_train = pd.DataFrame(df_train_scaled, columns=df_train.columns, index=df_train.index)

  df_test_scaled = scaler.transform(df_test) # Use transform, not fit_transform for test data
  df_test = pd.DataFrame(df_test_scaled, columns=df_test.columns, index=df_test.index)

  df_predict_scaled = scaler.transform(df_predict)
  df_predict = pd.DataFrame(df_predict_scaled, columns=df_predict.columns, index=df_predict.index)

  return df_train, y_train, df_test, y_test, df_predict, test_date, list_date

In [ ]:
# Feature Engineering and data processing pipeline

ticker = '^SPX'
start_date = '2020-01-01'
test_start_date = '2026-01-01'
current_date = '2026-06-30'
label = 'binary'

X_train_spx, y_train_spx, X_test_spx, y_test_spx, df_predict_spx, test_date_spx, list_date_spx = prep_data(ticker, start_date, test_start_date, current_date, label=label)
X_train_spx.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,Monday,Thursday,Tuesday,Wednesday,month,sin_month,cos_month,ratio close to high T0,ratio close to low T0,ratio close to open T0,...,Bearish Marubozu T-2,Spinning_Tops_Doji T0,Spinning_Tops_Doji T-1,Spinning_Tops_Doji T-2,Bullish Hammer T0,Bearish Shooting Star T0,Bullish Harami T0,Bearish Harami T0,Bullish Engulfing T0,Bearish Engulfing T0
129,-0.478597,-0.500227,-0.510404,1.963577,0.050871,-0.615624,-1.244802,0.776264,0.693529,0.567808,...,0.183666,-0.338812,-0.338812,-0.538508,-0.348049,-0.355857,-0.111721,0.0,-0.060324,-0.149126
130,-0.478597,1.999094,-0.510404,-0.509275,0.050871,-0.615624,-1.244802,-0.520951,0.856571,-0.877886,...,0.183666,-0.338812,-0.338812,-0.538508,-0.348049,-0.355857,-0.111721,0.0,-0.060324,-0.149126
131,-0.478597,-0.500227,-0.510404,-0.509275,0.050871,-0.615624,-1.244802,0.780712,1.483074,1.124148,...,0.183666,-0.338812,-0.338812,-0.538508,-0.348049,-0.355857,-0.111721,0.0,-0.060324,-0.149126
132,2.089440,-0.500227,-0.510404,-0.509275,0.050871,-0.615624,-1.244802,-3.078501,-0.722357,-1.767191,...,0.183666,-0.338812,-0.338812,-0.538508,-0.348049,-0.355857,-0.111721,0.0,-0.060324,-0.149126
133,-0.478597,-0.500227,1.959234,-0.509275,0.050871,-0.615624,-1.244802,0.698905,2.570631,1.975973,...,0.183666,-0.338812,-0.338812,1.856983,-0.348049,-0.355857,-0.111721,0.0,-0.060324,-0.149126


# Sample Autogluon Code

In [ ]:
# Show Sample Autogluon Code
# Prepare autogluon datasets
auto_train_data = pd.concat([X_train_spx, y_train_spx], axis=1)
auto_test_data = pd.concat([X_test_spx, y_test_spx], axis=1)

# Initialize and train the predictor
predictor = TabularPredictor(
      label='cat_close',
      learner_kwargs={"random_state": 42},
      eval_metric='accuracy'  # 'roc_auc', 'f1', 'r2', 'root_mean_squared_error'
).fit(
      train_data=auto_train_data,
      time_limit=600,         # Stop training after X seconds (e.g., 10 minutes)
      #presets='best_quality' # Options include 'medium_quality', 'best_quality', 'high_quality'
  )

No path specified. Models will be saved in: "AutogluonModels/ag-20261115_144303"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.18
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct 22 03:07:54 UTC 2026
CPU Count:          2
Pytorch Version:    2.13.0+cu130
CUDA Version:       CUDA is not available
Memory Avail:       10.51 GB / 12.67 GB (82.9%)
Disk Space Avail:   102.27 GB / 107.72 GB (94.9%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on datasets <100000 samples by using Tabu

In [ ]:
# Sample Leaderboard
predictor.leaderboard(auto_test_data)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,XGBoost,0.550000,0.550725,accuracy,0.014781,0.007771,11.729249,0.014781,0.007771,11.729249,1,True,9
1,LightGBM,0.541667,0.557971,accuracy,0.003689,0.003324,10.899690,0.003689,0.003324,10.899690,1,True,2
2,LightGBMXT,0.541667,0.550725,accuracy,0.004800,0.004050,5.380348,0.004800,0.004050,5.380348,1,True,1
3,NeuralNetTorch,0.541667,0.543478,accuracy,0.086246,0.146300,11.933274,0.086246,0.146300,11.933274,1,True,10
4,WeightedEnsemble_L2,0.525000,0.590580,accuracy,0.298919,0.229060,77.669243,0.005001,0.001157,0.101119,2,True,12
5,NeuralNetFastAI,0.516667,0.557971,accuracy,0.023416,0.028731,4.940163,0.023416,0.028731,4.940163,1,True,8
6,RandomForestGini,0.516667,0.565217,accuracy,0.113885,0.088438,4.807777,0.113885,0.088438,4.807777,1,True,3
7,LightGBMLarge,0.508333,0.557971,accuracy,0.037865,0.015021,55.154600,0.037865,0.015021,55.154600,1,True,11
8,ExtraTreesEntr,0.508333,0.525362,accuracy,0.123553,0.089007,1.528074,0.123553,0.089007,1.528074,1,True,7
9,RandomForestEntr,0.500000,0.532609,accuracy,0.106595,0.088311,8.512787,0.106595,0.088311,8.512787,1,True,4


In [ ]:
# Instantiate the traditional ML Models
rs = 42

mdl_xgb = XGBClassifier(random_state=rs)
mdl_lr = LogisticRegression(random_state=rs, l1_ratio=0.5, max_iter=5000)
mdl_gnb = GaussianNB()
mdl_lgb = lgb.LGBMClassifier(random_state=rs)
mdl_cat = CatBoostClassifier(random_state=rs, verbose=0)
mdl_dt = DecisionTreeClassifier(random_state=rs)
mdl_rf = RandomForestClassifier(random_state=rs)
mdl_knn = KNeighborsClassifier()

In [ ]:
# Prepare function which will be used subsequently for training models and generating performance leaderboard

model_name_list = ['XGBoost', 'Logistic Regression', 'Gaussian NB', 'LGBM', 'Catboost', 'Decision Tree', 'Random Forest', 'KNN']
model_list = [mdl_xgb, mdl_lr, mdl_gnb, mdl_lgb, mdl_cat, mdl_dt, mdl_rf, mdl_knn]

# Prepare function to train and evaluate models
def train_and_evaluate_models(model_name_list, model_list, X_train, y_train, X_test, y_test, cat = 'binary'):
  model_source = []
  accuracy_list = []
  precision_list = []
  recall_list = []
  #roc_auc_list = []
  pnl_list = []

  def multiclass_preds_to_binary(y_pred, pos_classes=[2, 3]):
    """
    Maps discrete multi-class array to binary (0 or 1).
    pos_classes: List of multi-class integers that represent the positive binary class (1).
    """
    return np.isin(y_pred, pos_classes).astype(int)

  # Traditional ML
  for i in model_list:
    i.fit(X_train, y_train)
    y_pred = i.predict(X_test)
    #predicted_proba = i.predict_proba(X_test)[:, 1]

    if cat == 'multi':
      y_pred = multiclass_preds_to_binary(y_pred, pos_classes=[2, 3])
      y_test_converted = multiclass_preds_to_binary(y_test, pos_classes=[2, 3])
    else:
      y_test_converted = y_test

    accuracy_list.append(accuracy_score(y_test_converted, y_pred))
    precision_list.append(precision_score(y_test_converted, y_pred))
    recall_list.append(recall_score(y_test_converted, y_pred))
    #roc_auc_list.append(round(roc_auc_score(y_test_converted, predicted_proba),4))

    cumulative_PnL = [x == y for x, y in zip(y_pred, y_test_converted)]
    cumulative_PnL = sum([50 if i == True else -400 for i in cumulative_PnL])
    pnl_list.append(cumulative_PnL)
    model_source.append('Traditional ML')

  # Autogluon
  # Prepare autogluon datasets
  auto_train_data = pd.concat([X_train, y_train], axis=1)
  auto_test_data = pd.concat([X_test, y_test], axis=1)

  # Define the target column you want to predict
  label_column = 'cat_close'

  # Initialize and train the predictor
  predictor = TabularPredictor(
      label=label_column,
      learner_kwargs={"random_state": 42},
      eval_metric='accuracy'  # Optional: e.g., 'roc_auc', 'f1', 'r2', 'root_mean_squared_error'
  ).fit(
      train_data=auto_train_data,
      time_limit=600,         # Optional: Stop training after X seconds (e.g., 10 minutes)
      #presets='best_quality'  # Optional: Options include 'medium_quality', 'best_quality', 'high_quality'
  )
  for i in predictor.model_names():
    model_name_list.append(i)
    y_pred = predictor.predict(X_test, model=i)
    #predicted_proba = predictor.predict_proba(X_test, model=i)[1]

    if cat == 'multi':
      y_pred = multiclass_preds_to_binary(y_pred, pos_classes=[2, 3])
      y_test_converted = multiclass_preds_to_binary(y_test, pos_classes=[2, 3])
    else:
      y_test_converted = y_test

    accuracy_list.append(accuracy_score(y_test_converted, y_pred))
    precision_list.append(precision_score(y_test_converted, y_pred))
    recall_list.append(recall_score(y_test_converted, y_pred))
    #roc_auc_list.append(round(roc_auc_score(y_test, predicted_proba),4))

    cumulative_PnL = [x == y for x, y in zip(y_pred, y_test_converted)]
    cumulative_PnL = sum([50 if i == True else -400 for i in cumulative_PnL])
    pnl_list.append(cumulative_PnL)
    model_source.append('Autogluon')
  df_mdl_performance = pd.DataFrame({'Model Source': model_source, 'Model': model_name_list, 'Accuracy': accuracy_list, 'Precision': precision_list, 'Recall': recall_list, 'PnL' : pnl_list}) # 'ROC AUC': roc_auc_list,
  return df_mdl_performance.sort_values(by='Accuracy', ascending=False)

# Training Model for SPX ticker

In [ ]:
model_name_list = ['XGBoost', 'Logistic Regression', 'Gaussian NB', 'LGBM', 'Catboost', 'Decision Tree', 'Random Forest', 'KNN']
df_mdl_performance = train_and_evaluate_models(model_name_list, model_list, X_train_spx, y_train_spx, X_test_spx, y_test_spx, cat='binary')

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 748, number of negative: 631
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006869 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 70744
[LightGBM] [Info] Number of data points in the train set: 1379, number of used features: 304
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.542422 -> initscore=0.170097
[LightGBM] [Info] Start training from score 0.170097


No path specified. Models will be saved in: "AutogluonModels/ag-20261118_135005"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.18
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct 22 03:07:54 UTC 2026
CPU Count:          2
Pytorch Version:    2.13.0+cu130
CUDA Version:       CUDA is not available
Memory Avail:       10.71 GB / 12.67 GB (84.5%)
Disk Space Avail:   102.27 GB / 107.72 GB (94.9%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on datasets <100000 samples by using Tabu

In [ ]:
df_mdl_performance

,Model Source,Model,Accuracy,Precision,Recall,PnL
5,Traditional ML,Decision Tree,0.608333,0.660714,0.569231,-15150
8,Autogluon,LightGBMXT,0.575000,0.560345,1.000000,-16950
19,Autogluon,WeightedEnsemble_L2,0.566667,0.565657,0.861538,-17400
7,Traditional ML,KNN,0.566667,0.591549,0.646154,-17400
9,Autogluon,LightGBM,0.558333,0.561224,0.846154,-17850
16,Autogluon,XGBoost,0.550000,0.546218,1.000000,-18300
17,Autogluon,NeuralNetTorch,0.541667,0.541667,1.000000,-18750
3,Traditional ML,LGBM,0.516667,0.537634,0.769231,-20100
18,Autogluon,LightGBMLarge,0.516667,0.539326,0.738462,-20100
15,Autogluon,NeuralNetFastAI,0.516667,0.533333,0.861538,-20100


# Training Models for NDX ticker

In [ ]:

ticker = '^NDX'
start_date = '2020-01-01'
test_start_date = '2026-01-01'
current_date = '2026-06-30'

X_train_ndx, y_train_ndx, X_test_ndx, y_test_ndx, df_predict_ndx, test_date_ndx, list_date_ndx = prep_data(ticker, start_date, test_start_date, current_date, label=label)
model_name_list = ['XGBoost', 'Logistic Regression', 'Gaussian NB', 'LGBM', 'Catboost', 'Decision Tree', 'Random Forest', 'KNN']
df_mdl_performance_ndx = train_and_evaluate_models(model_name_list, model_list, X_train_ndx, y_train_ndx, X_test_ndx, y_test_ndx)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
No path specified. Models will be saved in: "AutogluonModels/ag-20261116_143112"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.18
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct 22 

In [ ]:
df_mdl_performance_ndx

,Model Source,Model,Accuracy,Precision,Recall,PnL
19,Autogluon,WeightedEnsemble_L2,0.650000,0.630000,0.926471,-12900
6,Traditional ML,Random Forest,0.633333,0.639535,0.808824,-13800
17,Autogluon,NeuralNetTorch,0.625000,0.623656,0.852941,-14250
8,Autogluon,LightGBMXT,0.591667,0.581197,1.000000,-16050
12,Autogluon,CatBoost,0.591667,0.594059,0.882353,-16050
16,Autogluon,XGBoost,0.566667,0.566667,1.000000,-17400
9,Autogluon,LightGBM,0.566667,0.566667,1.000000,-17400
10,Autogluon,RandomForestGini,0.566667,0.588889,0.779412,-17400
18,Autogluon,LightGBMLarge,0.566667,0.566667,1.000000,-17400
14,Autogluon,ExtraTreesEntr,0.550000,0.579545,0.750000,-18300


# Training Model for DJI ticker

In [ ]:

ticker = '^DJI'
start_date = '2020-01-01'
test_start_date = '2026-01-01'
current_date = '2026-06-30'

X_train_dji, y_train_dji, X_test_dji, y_test_dji, df_predict_dji, test_date_dji, list_date_dji = prep_data(ticker, start_date, test_start_date, current_date, label=label)
model_name_list = ['XGBoost', 'Logistic Regression', 'Gaussian NB', 'LGBM', 'Catboost', 'Decision Tree', 'Random Forest', 'KNN']
df_mdl_performance_dji = train_and_evaluate_models(model_name_list, model_list, X_train_dji, y_train_dji, X_test_dji, y_test_dji)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
No path specified. Models will be saved in: "AutogluonModels/ag-20261116_143527"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.18
Operating System:   Li

In [ ]:
df_mdl_performance_dji

,Model Source,Model,Accuracy,Precision,Recall,PnL
4,Traditional ML,Catboost,0.616667,0.619048,0.787879,-14700
11,Autogluon,RandomForestEntr,0.600000,0.612500,0.742424,-15600
1,Traditional ML,Logistic Regression,0.591667,0.607595,0.727273,-16050
6,Traditional ML,Random Forest,0.591667,0.616438,0.681818,-16050
0,Traditional ML,XGBoost,0.575000,0.594937,0.712121,-16950
8,Autogluon,LightGBMXT,0.566667,0.612903,0.575758,-17400
17,Autogluon,NeuralNetTorch,0.558333,0.554622,1.000000,-17850
3,Traditional ML,LGBM,0.558333,0.578313,0.727273,-17850
19,Autogluon,WeightedEnsemble_L2,0.558333,0.574713,0.757576,-17850
9,Autogluon,LightGBM,0.550000,0.558824,0.863636,-18300


#Training Model for SPX Multiclass ticker

In [ ]:
ticker = '^SPX'
start_date = '2020-01-01'
test_start_date = '2026-01-01'
current_date = '2026-06-30'
label = 'multi'

X_train_spx, y_train_spx, X_test_spx, y_test_spx, df_predict_spx, test_date_spx, list_date_spx = prep_data(ticker, start_date, test_start_date, current_date, label=label)

model_name_list = ['XGBoost', 'Logistic Regression', 'Gaussian NB', 'LGBM', 'Catboost', 'Decision Tree', 'Random Forest', 'KNN']
df_mdl_performance = train_and_evaluate_models(model_name_list, model_list, X_train_spx, y_train_spx, X_test_spx, y_test_spx, cat='multi')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
No path specified. Models will be saved in: "AutogluonModels/ag-20261116_144933"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.18
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct 22 

In [ ]:
df_mdl_performance

,Model Source,Model,Accuracy,Precision,Recall,PnL
13,Autogluon,CatBoost,0.550000,0.550459,0.923077,-18300
0,Traditional ML,XGBoost,0.541667,0.560976,0.707692,-18750
17,Autogluon,NeuralNetTorch,0.541667,0.541667,1.000000,-18750
7,Traditional ML,KNN,0.533333,0.578947,0.507692,-19200
6,Traditional ML,Random Forest,0.533333,0.551724,0.738462,-19200
10,Autogluon,LightGBM,0.516667,0.540230,0.723077,-20100
9,Autogluon,LightGBMXT,0.516667,0.542169,0.692308,-20100
19,Autogluon,WeightedEnsemble_L2,0.516667,0.532710,0.876923,-20100
12,Autogluon,RandomForestEntr,0.508333,0.531915,0.769231,-20550
4,Traditional ML,Catboost,0.508333,0.531915,0.769231,-20550


#Training Model for NDX Multiclass ticker

In [ ]:

ticker = '^NDX'
start_date = '2020-01-01'
test_start_date = '2026-01-01'
current_date = '2026-06-30'
label = 'multi'

X_train_ndx, y_train_ndx, X_test_ndx, y_test_ndx, df_predict_ndx, test_date_ndx, list_date_ndx = prep_data(ticker, start_date, test_start_date, current_date, label=label)
model_name_list = ['XGBoost', 'Logistic Regression', 'Gaussian NB', 'LGBM', 'Catboost', 'Decision Tree', 'Random Forest', 'KNN']
df_mdl_performance_ndx = train_and_evaluate_models(model_name_list, model_list, X_train_ndx, y_train_ndx, X_test_ndx, y_test_ndx, cat='multi')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
No path specified. Models will be saved in: "AutogluonModels/ag-20261116_150511"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.18
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct 22 

In [ ]:
df_mdl_performance_ndx

,Model Source,Model,Accuracy,Precision,Recall,PnL
12,Autogluon,RandomForestEntr,0.608333,0.615385,0.823529,-15150
13,Autogluon,CatBoost,0.600000,0.587719,0.985294,-15600
9,Autogluon,LightGBMXT,0.583333,0.595745,0.823529,-16500
6,Traditional ML,Random Forest,0.575000,0.604938,0.720588,-16950
14,Autogluon,ExtraTreesGini,0.575000,0.591398,0.808824,-16950
17,Autogluon,NeuralNetTorch,0.566667,0.567797,0.985294,-17400
18,Autogluon,LightGBMLarge,0.566667,0.566667,1.000000,-17400
19,Autogluon,WeightedEnsemble_L2,0.566667,0.567797,0.985294,-17400
11,Autogluon,RandomForestGini,0.550000,0.577778,0.764706,-18300
15,Autogluon,ExtraTreesEntr,0.541667,0.565657,0.823529,-18750


#Training Model for DJI Multiclass ticker

In [ ]:

ticker = '^DJI'
start_date = '2020-01-01'
test_start_date = '2026-01-01'
current_date = '2026-06-30'
label = 'multi'

X_train_dji, y_train_dji, X_test_dji, y_test_dji, df_predict_dji, test_date_dji, list_date_dji = prep_data(ticker, start_date, test_start_date, current_date, label=label)
model_name_list = ['XGBoost', 'Logistic Regression', 'Gaussian NB', 'LGBM', 'Catboost', 'Decision Tree', 'Random Forest', 'KNN']
df_mdl_performance_dji = train_and_evaluate_models(model_name_list, model_list, X_train_dji, y_train_dji, X_test_dji, y_test_dji, cat='multi')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
No path specified. Models will be saved in: "AutogluonModels/ag-20261116_151749"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.18
Operating System:   Li

In [ ]:
df_mdl_performance_dji

,Model Source,Model,Accuracy,Precision,Recall,PnL
1,Traditional ML,Logistic Regression,0.608333,0.609195,0.803030,-15150
11,Autogluon,RandomForestGini,0.608333,0.614458,0.772727,-15150
14,Autogluon,ExtraTreesGini,0.591667,0.595506,0.803030,-16050
12,Autogluon,RandomForestEntr,0.575000,0.584270,0.787879,-16950
15,Autogluon,ExtraTreesEntr,0.566667,0.574468,0.818182,-17400
6,Traditional ML,Random Forest,0.558333,0.580247,0.712121,-17850
4,Traditional ML,Catboost,0.558333,0.573034,0.772727,-17850
17,Autogluon,NeuralNetTorch,0.550000,0.550000,1.000000,-18300
18,Autogluon,LightGBMLarge,0.541667,0.566265,0.712121,-18750
13,Autogluon,CatBoost,0.541667,0.551402,0.893939,-18750
